> ## SOLUTION / ANSWER KEY &mdash; Lab 6.4
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-04-assemble-agent.ipynb`](../lab-04-assemble-agent.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 6.4 &mdash; Assemble an Agent with create_agent

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Bind the model + a tools list into one agent with create_agent
- See that the agent is a runnable CompiledStateGraph with model+tools nodes
- Invoke it on a real question and read the answer

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. Read the **Concept**, fill the real `___` blanks in **Build it** (real tool bodies, real prompts, the real `create_agent` call), then **Run it for real** &mdash; a real LLM drives a real agent over real tools &mdash; and **read the trace** to see exactly what it did. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`) and a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`). If Ollama is down, the run cells print how to start it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* aborts the whole agent run (you'll see exactly this in Lab 11).

**Reference:** [Module 6 slides &mdash; Assemble an agent](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"))   # SERPER_API_KEY, WOLFRAM_ALPHA_APPID, GROQ/OPENAI keys

WORK = "/tmp/biaa-lab-06-04"
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model. Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
A LangChain (v1) agent is a **model** + a **tools** list bound together by
**`create_agent(model, tools)`** (from `langchain.agents`). It returns a runnable **`CompiledStateGraph`**
&mdash; a small graph with a **`model`** node (decide) and a **`tools`** node (act) that loop until the
model answers. You assemble it here, then run it for real.

In [ ]:
from langchain_core.tools import tool

@tool
def lookup(key: str) -> str:
    """Look up a known fact by key, e.g. 'capital of france'."""
    return {"capital of france": "Paris", "capital of japan": "Tokyo"}.get(key.lower().strip(), "unknown")

print("one tool ready:", lookup.name)

## Build it
Assemble the agent in `build_agent`: gather the tools list and bind them to the real `llm` with
`create_agent`.

In [ ]:
from langchain.agents import create_agent

def build_agent():
    tools = [lookup]
    return create_agent(llm, tools)

try:
    agent = build_agent()
    print("agent type  :", type(agent).__name__)
    print("graph nodes :", set(agent.nodes) - {"__start__"})
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Invoke the assembled agent on a real question and read the final answer (and the tool call it made to get there).

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        result = agent.invoke({"messages": [("user", "What is the capital of Japan?")]},
                              config={"recursion_limit": 6})
        print_trace(result)
        print("---")
        print("final answer:", result["messages"][-1].content)
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- `type(agent).__name__` is **`CompiledStateGraph`** &mdash; a real runnable graph, not a wrapper.
- Its nodes are **`model`** and **`tools`**: the loop is decide -> act -> decide until an answer.
- The final message's `.content` is the answer; everything before it is how the agent got there.

## Your turn (open task &mdash; no grader)
Add a second tool (a calculator, or a second fact source) to `build_agent`'s list and re-run with a
question that needs both. **What good looks like:** the trace shows two different `TOOL CALL`s and the
final answer combines them &mdash; a real multi-tool agent, assembled in three lines.

---
### Key takeaway
model + tools -> create_agent = a runnable CompiledStateGraph that loops decide<->act until it answers. Next: read the full trace and cap the loop.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>